## Setup — run this first

Mounts Drive, points the notebook at your project folder, and installs
what's missing. No git, no tokens.

**Your Drive folder must look like this:**

```
MyDrive/Ghana_Dropout_Project_R02/
├── config.py          <- these three at the TOP level,
├── losses.py             not inside notebooks/
├── pipeline.py
├── requirements.txt
├── notebooks/         <- the 11 notebooks
└── data-raw/
    └── ghana_dropout_study_M.xlsx
```

`results/`, `figures/`, `models/` and `data-processed/` are created for you.

Drive saves as it goes, so there is nothing to push — but see the checklist
in the last cell before you submit.


In [16]:
# ============================================================
# SETUP — Google Drive. Run first. Safe to re-run.
# ============================================================
import os, sys, subprocess
from pathlib import Path

PROJECT = "/content/drive/MyDrive/Ghana_Dropout_Project_R02"   # <-- edit if yours differs
RAW_XLSX_NAME = "ghana_dropout_study_M.xlsx"

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")

    root = Path(PROJECT)
    if not root.exists():
        raise FileNotFoundError(
            f"{PROJECT} does not exist.\n"
            "Create that folder in My Drive and put config.py, losses.py, "
            "pipeline.py, requirements.txt, the notebooks/ folder and "
            "data-raw/ inside it."
        )

    # the three modules must sit at the project root, not in notebooks/
    missing = [m for m in ("config.py", "losses.py", "pipeline.py")
               if not (root / m).exists()]
    if missing:
        stray = [m for m in missing if (root / "notebooks" / m).exists()]
        msg = f"Missing from {PROJECT}: {missing}"
        if stray:
            msg += (f"\n{stray} are in notebooks/ instead. Move them UP one "
                    "level, into the project folder itself. If they stay in "
                    "notebooks/, that folder gets treated as the project root "
                    "and results/ is written in the wrong place.")
        raise FileNotFoundError(msg)

    os.chdir(root)
    os.environ["DROPOUT_REPO"] = str(root)

    # Forget any previously loaded copy of the project modules. Python keeps
    # the first version it imported for the whole session, so an edited
    # config.py is silently ignored until the runtime restarts. This makes
    # every run use the files currently in Drive.
    for _m in ("config", "losses", "pipeline"):
        sys.modules.pop(_m, None)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

    # ---- dependencies: only install what is actually missing ------------
    need = []
    for mod, pkg in [("lightgbm", "lightgbm"), ("shap", "shap"),
                     ("catboost", "catboost"), ("xgboost", "xgboost"),
                     ("imblearn", "imbalanced-learn"), ("openpyxl", "openpyxl")]:
        try:
            __import__(mod)
        except ImportError:
            need.append(pkg)
    if need:
        print("installing:", need)
        subprocess.run(f"pip install -q {' '.join(need)}", shell=True)
    else:
        print("all dependencies present")

    # ---- raw workbook ---------------------------------------------------
    (root / "data-raw").mkdir(exist_ok=True)
    xlsx = root / "data-raw" / RAW_XLSX_NAME
    if xlsx.exists():
        print(f"raw workbook: {xlsx.name}")
    else:
        loose = list(root.glob(RAW_XLSX_NAME)) + list(root.glob(f"**/{RAW_XLSX_NAME}"))
        if loose:
            import shutil
            shutil.copy(loose[0], xlsx)
            print(f"copied {loose[0]} -> data-raw/")
        else:
            print(f"NOT FOUND: data-raw/{RAW_XLSX_NAME}\n"
                  "Notebook 1 needs it. Notebooks 2-9 read "
                  "data-processed/cleaned_data.csv instead and are fine "
                  "without it.")

    print(f"\nPROJECT : {os.getcwd()}")
else:
    print("Not in Colab — paths resolve from the project root.")


all dependencies present
raw workbook: ghana_dropout_study_M.xlsx

PROJECT : /content/drive/MyDrive/Ghana_Dropout_Project_R02


# Notebook 1 — Data Cleaning

**Output:** `data-processed/cleaned_data.csv` plus a reconciled column
cascade and a missingness table.

## What changed from R01, and why

| Change | Reason |
|---|---|
| No `drive.mount()`, no hard-coded `PROJECT_DIR` | Q4 — no R01 notebook ran against a fresh clone |
| Identifier drop matches **whole tokens**, not substrings | `"id" in "residence_type"` is True. The R01 rule silently dropped any column containing those two letters |
| **No imputation, no encoding, no scaling here** | GATE-1(i). Every fitted transform now happens inside the fold (`pipeline.py`). This notebook only does operations that use fixed constants |
| Near-unique pruning restricted to text columns | The R01 rule dropped any column with >98% unique values, which targets IDs but catches continuous variables. A float attendance rate with 990 distinct values would have been deleted |
| Column cascade printed and committed | Q2, Q11 — nobody could reproduce 44 → 41 |
| Missingness and out-of-range counts computed | Q11 — M4 says these "were not formally recorded". They are recoverable |
| Leakage list consolidated into `config.LEAKAGE_EXACT` | R01 dropped leakage columns twice, in cells 14 and 22, with two different lists |

Cleaning is deliberately **conservative**: it removes only what cannot be a
predictor (identifiers, leakage, empty columns) and repairs only what is
unambiguously an error. Everything judgement-laden happens in the fold.

In [17]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [18]:
from config import *
from pipeline import binarise_target

banner("NOTEBOOK 1 — DATA CLEANING")
OUT = run_dir("notebook01_cleaning")
capture_environment(OUT)
print("outputs ->", OUT)

if RAW_WORKBOOK is None:
    raise FileNotFoundError(
        "Raw workbook not found. Expected data-raw/ghana_dropout_study_M.xlsx.\n"
        "Pupil-level data is not committed (ethics, M5), so place it there "
        "locally or set DROPOUT_REPO to a folder that contains it."
    )

df = pd.read_excel(RAW_WORKBOOK)
N_RAW_ROWS, N_RAW_COLS = df.shape
print(f"raw: {N_RAW_ROWS} rows x {N_RAW_COLS} columns   <-- Table 1 'Size (raw)'")

NOTEBOOK 1 — DATA CLEANING
repo            : /content/drive/MyDrive/Ghana_Dropout_Project_R02
provenance      : NONE — set FREEZE_TAG in config.py before scoring the test set
school_handling : drop
FEATURE SET     : records   (PRIMARY — school records only)
primary metric  : auc_pr
outputs -> /content/drive/MyDrive/Ghana_Dropout_Project_R02/results/notebook01_cleaning/20260921T134433Z_records
raw: 1000 rows x 48 columns   <-- Table 1 'Size (raw)'


In [19]:
# ---- 1. normalise column names ------------------------------------------
df.columns = (pd.Index(df.columns).astype(str)
              .str.strip().str.lower()
              .str.replace(r"\s+", "_", regex=True)
              .str.replace(r"[^a-z0-9_]", "", regex=True))
df = df.loc[:, ~df.columns.duplicated()]

# ---- 2. duplicate rows ---------------------------------------------------
n_dup = int(df.duplicated().sum())
df_with_dups = df.copy()            # kept for the Section 3c diagnostic
print(f"duplicate rows: {n_dup}  (identical in EVERY column, study_id included)")
if n_dup:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"  removed -> {len(df)} rows")
    print("  The same pupil entered twice, not two pupils who happen to "
          "match: study_id is identical too.")

assert TARGET in df.columns, f"target '{TARGET}' not found. Columns: {list(df.columns)}"

duplicate rows: 19  (identical in EVERY column, study_id included)
  removed -> 981 rows
  The same pupil entered twice, not two pupils who happen to match: study_id is identical too.


In [20]:
# ---- 3. THE COLUMN CASCADE (Q2, Q11) ------------------------------------
# Precedence: leakage > identifier > empty, so the reason a reader needs is
# never hidden behind an accident of ordering.

cascade, n = [], df.shape[1]
cascade.append({"step": "0. raw workbook", "criterion": "-",
                "n_dropped": 0, "n_remaining": n, "columns": ""})

reasons = {c: drop_reason(c) for c in df.columns if c != TARGET}
leak    = [c for c, r in reasons.items() if r == "leakage"]
ident   = [c for c, r in reasons.items() if r and r.startswith("identifier")]
temporal = [c for c, r in reasons.items() if r and r.startswith("temporal leakage")]
measure  = [c for c, r in reasons.items() if r and r.startswith("differential measurement")]
derived = [c for c, r in reasons.items() if r and r.startswith("derived duplicate")]
suspect = [c for c, r in reasons.items() if r and r.startswith("failed range")]
empty   = [c for c in df.columns
           if c != TARGET and c not in leak + temporal + measure + ident + derived + suspect
           and df[c].isna().all()]

for step, crit, cols in [
    ("1. target leakage", "derived from or dated by the outcome", leak),
    ("1b. temporal leakage", "recorded after the outcome occurred", temporal),
    ("1c. differential measurement", "answered by friends for dropouts, by pupils for stayers", measure),
    ("2. identifier / provenance", "exact name or whole underscore token", ident),
    ("3. derived duplicate", "arithmetically recoverable from retained columns", derived),
    ("4. failed range validation", "values outside the declared scale", suspect),
    ("5. entirely empty", "100% missing", empty),
]:
    n -= len(cols)
    cascade.append({"step": step, "criterion": crit, "n_dropped": len(cols),
                    "n_remaining": n, "columns": "; ".join(cols)})

# text columns that are near-unique are identifiers the keyword rule missed.
# NOTE: restricted to text. The R01 rule applied to every column, so a
# continuous variable with many distinct values could have been deleted.
text_cols = [c for c in df.columns
             if c not in leak + temporal + measure + ident + derived + suspect + empty and c != TARGET
             and is_text(df[c])]
near_unique = [c for c in text_cols if df[c].nunique(dropna=True) > 0.98 * len(df)]
n -= len(near_unique)
cascade.append({"step": "6. near-unique text", "criterion": ">98% distinct values, text only",
                "n_dropped": len(near_unique), "n_remaining": n,
                "columns": "; ".join(near_unique)})
cascade.append({"step": "7. constant / near-constant", "criterion": "pruned IN-FOLD on training statistics",
                "n_dropped": np.nan, "n_remaining": np.nan,
                "columns": "fold-dependent — see pipeline.preprocess_inside_fold"})

cascade_df = pd.DataFrame(cascade)
cascade_df.to_csv(OUT / "column_cascade.csv", index=False)
print(cascade_df[["step", "criterion", "n_dropped", "n_remaining"]].to_string(index=False))
print("\nreasons, column by column:")
for c in leak + temporal + measure + ident + derived + suspect + near_unique:
    print(f"  {c:42s} <- {reasons.get(c, 'near-unique text')}")

# NOTE: derived duplicates and range-failed columns stay in cleaned_data.csv
# so a reader can inspect them; pipeline.preprocess_inside_fold() excludes
# them from every feature matrix via the same drop_reason() call.
df = df.drop(columns=[c for c in leak + ident + empty + near_unique
                      if c in df.columns], errors="ignore")
print(f"\nretained in cleaned_data.csv but EXCLUDED from every model: "
      f"{temporal + measure + derived + [c for c in suspect]}")
print(f"\nafter cascade: {df.shape[1]} columns (incl. target)")

                        step                                               criterion  n_dropped  n_remaining
             0. raw workbook                                                       -        0.0         48.0
           1. target leakage                    derived from or dated by the outcome        1.0         47.0
        1b. temporal leakage                     recorded after the outcome occurred        1.0         46.0
1c. differential measurement answered by friends for dropouts, by pupils for stayers       23.0         23.0
  2. identifier / provenance                    exact name or whole underscore token        4.0         19.0
        3. derived duplicate        arithmetically recoverable from retained columns        1.0         18.0
  4. failed range validation                       values outside the declared scale        0.0         18.0
           5. entirely empty                                            100% missing        1.0         17.0
         6. near-un

In [21]:
# ---- 3b. what the OLD substring rule would have removed -----------------
OLD_KEYWORDS = ["id", "student_id", "study_id", "record_id", "serial",
                "index", "registration", "date_recorded", "enumerator_initials"]
raw_names = pd.read_excel(RAW_WORKBOOK, nrows=0).columns
raw_names = (pd.Index(raw_names).astype(str).str.strip().str.lower()
             .str.replace(r"\s+", "_", regex=True)
             .str.replace(r"[^a-z0-9_]", "", regex=True))

old_drop = {c for c in raw_names
            if any(k in c for k in OLD_KEYWORDS) and c != TARGET}
restored = sorted(old_drop - set(leak) - set(temporal) - set(measure) - set(ident) - set(derived) - set(suspect))
if restored:
    print("*** RESTORED — the R01 substring rule dropped these silently: ***")
    for c in restored:
        print("   ", c)
    pd.DataFrame({"restored_column": restored}).to_csv(
        OUT / "columns_restored_vs_substring_rule.csv", index=False)
    print("\nThese are in the feature matrix now, so the matrix differs from "
          "every previously reported table. State it in M6.")
else:
    print("No column was lost to the R01 substring rule on this data.")

No column was lost to the R01 substring rule on this data.


In [22]:
# ---- 4. target to 0/1 ----------------------------------------------------
before = df[TARGET].value_counts(dropna=False)
df[TARGET] = binarise_target(df[TARGET])
unmapped = int(df[TARGET].isna().sum())
print("target, as recorded:\n", before.to_string())
if unmapped:
    raise ValueError(
        f"{unmapped} target values did not map. Add them to "
        f"pipeline.TARGET_LABEL_MAP rather than dropping the rows silently."
    )
df[TARGET] = df[TARGET].astype(int)
n_pos = int(df[TARGET].sum())
print(f"\ndropout {n_pos} / {len(df)} = {100*n_pos/len(df):.1f}%   "
      "<-- the base rate that must print beside every headline number")

target, as recorded:
 dropout_label
0 - Retained    895
1 - Dropout      86

dropout 86 / 981 = 8.8%   <-- the base rate that must print beside every headline number


In [23]:
# ---- 5. MISSINGNESS (Q11) ------------------------------------------------
miss = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "n_missing": [int(df[c].isna().sum()) for c in df.columns]})
miss["pct_missing"] = (100 * miss["n_missing"] / len(df)).round(2)
miss = miss.sort_values("n_missing", ascending=False)
miss.to_csv(OUT / "missingness_by_column.csv", index=False)

rows_any = int(df.isna().any(axis=1).sum())
print(f"records with >=1 missing value : {rows_any}/{len(df)} "
      f"({100*rows_any/len(df):.1f}%)   <-- M4 says this was never recorded")
print(f"columns with any missing       : {int((miss['n_missing']>0).sum())}")
print(miss[miss['n_missing'] > 0].to_string(index=False))

# The examiner's specific question: is the strongest feature imputed?
for c in ["average_exam_score", *ATTENDANCE_COLS]:
    if c in df.columns:
        print(f"  {c:26s} {100*df[c].isna().mean():5.1f}% missing "
              "(median-imputed in-fold)")

plt.figure(figsize=(10, 5))
top = miss[miss["n_missing"] > 0].head(20)
if len(top):
    sns.barplot(data=top, y="column", x="pct_missing", color="steelblue")
    plt.xlabel("% missing"); plt.title("Missingness by column")
    plt.tight_layout(); plt.savefig(OUT / "figures/missingness.png", dpi=200)
plt.close()

records with >=1 missing value : 266/981 (27.1%)   <-- M4 says this was never recorded
columns with any missing       : 5
                    column   dtype  n_missing  pct_missing
extracurricular_activities  object        244        24.87
 daily_study_hours_at_home  object         31         3.16
  no_of_siblings_in_school  object          9         0.92
       class_participation float64          2         0.20
         term_2_attendance float64          1         0.10
  average_exam_score           0.0% missing (median-imputed in-fold)
  term_1_attendance            0.0% missing (median-imputed in-fold)
  term_2_attendance            0.1% missing (median-imputed in-fold)
  term_3_attendance            0.0% missing (median-imputed in-fold)


In [24]:
# ---- 6. OUT-OF-RANGE VALUES (Q11) ---------------------------------------
# Table 2 shows term_1_attendance max 100.5 and term_2_attendance max 102.9.
# Attendance above 100% is a data error. R01 passed it to the model untreated.
rows = []
for c in ATTENDANCE_COLS:
    if c not in df.columns:
        continue
    s = pd.to_numeric(df[c], errors="coerce")
    rows.append({"column": c, "min": s.min(), "max": s.max(),
                 "n_above_100": int((s > ATTENDANCE_MAX).sum()),
                 "n_below_0": int((s < 0).sum()),
                 "n_exactly_0": int((s == 0).sum()),
                 "n_missing": int(s.isna().sum())})
out_df = pd.DataFrame(rows)
out_df.to_csv(OUT / "attendance_range_audit.csv", index=False)
print(out_df.to_string(index=False))
print(f"\nTOTAL out-of-range attendance values: "
      f"{int(out_df['n_above_100'].sum() + out_df['n_below_0'].sum())}")
print("\nTreatment: clipped to [0, 100] INSIDE the fold (pipeline.py step 4), "
      "using the physical bound rather than a data-derived threshold, so no "
      "information crosses the partition. Report the counts above in M6.")
print("\nNOTE: the R01 composite builder divided attendance by 100 only when "
      "the maximum exceeded 1.0, so values above 100 produced a NEGATIVE risk "
      "contribution and propagated into attendance_risk_index. Clipping first "
      "removes that.")

           column  min   max  n_above_100  n_below_0  n_exactly_0  n_missing
term_1_attendance 20.0 100.5            1          0            0          0
term_2_attendance 10.0 102.9            2          0            0          1
term_3_attendance  0.0  99.0            0          0           48          0

TOTAL out-of-range attendance values: 3

Treatment: clipped to [0, 100] INSIDE the fold (pipeline.py step 4), using the physical bound rather than a data-derived threshold, so no information crosses the partition. Report the counts above in M6.

NOTE: the R01 composite builder divided attendance by 100 only when the maximum exceeded 1.0, so values above 100 produced a NEGATIVE risk contribution and propagated into attendance_risk_index. Clipping first removes that.


## Section 3c — Data integrity diagnostics

Four checks that decide what the R01 results actually measured. None fits a
model; they audit the data. They look at the full dataset because the
question is about **when and how the data were recorded**, not predictive
value, so this is not the analyst-peeking problem Notebook 2 guards against.

In [25]:
from sklearn.model_selection import train_test_split
diag_rows = []

# ---------------------------------------------------------------------
# (a) Is term 3 attendance measured AFTER the pupil left?
# ---------------------------------------------------------------------
# If dropout happened during the year, a pupil who left in term 2 has term 3
# attendance of zero BY CONSTRUCTION. Then term_3_attendance is not a
# predictor of dropout, it is a record of it. attendance_risk_index gives
# term 3 its largest weight (0.45), so the composite would be built mostly
# on the outcome.
t3 = pd.to_numeric(df["term_3_attendance"], errors="coerce")
y = df[TARGET]
zero = t3 == 0
ct = pd.crosstab(zero.map({True: "term 3 = 0%", False: "term 3 > 0%"}),
                 y.map({0: "retained", 1: "dropout"}), margins=True)
print("(a) TERM 3 ATTENDANCE OF ZERO vs OUTCOME")
print(ct.to_string())
n_zero = int(zero.sum())
n_zero_drop = int((zero & (y == 1)).sum())
n_drop = int((y == 1).sum())
print(f"\n{n_zero_drop} of {n_zero} zero-attendance pupils are dropouts "
      f"({100*n_zero_drop/max(n_zero,1):.0f}%)")
print(f"{n_zero_drop} of {n_drop} dropouts have zero term-3 attendance "
      f"({100*n_zero_drop/max(n_drop,1):.0f}%)")
print("\ndropout rate by term-3 attendance band:")
for lo, hi in [(0, 0), (0.1, 20), (20, 50), (50, 80), (80, 101)]:
    m = (t3 >= lo) & (t3 <= hi)
    if m.sum():
        print(f"   term 3 in [{lo:>5}, {hi:>5}]: n={int(m.sum()):4d}  "
              f"dropout rate {100*y[m].mean():5.1f}%")

if n_zero and n_zero_drop / n_zero > 0.8:
    print("""
*** STRONG SIGNAL OF TEMPORAL LEAKAGE ***
Nearly every pupil with zero term-3 attendance is a dropout. That is what you
would see if term 3 was recorded after they had already left. If so,
term_3_attendance measures the outcome rather than predicting it, and a large
part of the 0.99 AUC-PR is the model reading the answer.

This is a question for whoever designed the instrument, not a coding
question: WHEN was dropout status recorded relative to term 3? Does "dropout"
mean "left during 2024/25" (term 3 is contaminated) or "did not return for
2025/26" (term 3 is legitimately prior)?

It is also the most plausible answer to the examiner's question, "what is
carrying that separation?", and it would be the most important sentence in
your Discussion.""")
diag_rows.append({"check": "term3_zero_vs_dropout", "n_zero": n_zero,
                  "n_zero_dropout": n_zero_drop, "n_dropout": n_drop,
                  "pct_zero_that_dropout": round(100*n_zero_drop/max(n_zero, 1), 1)})

# ---------------------------------------------------------------------
# (b) social_studies_exam_score: two scales, or entry errors?
# ---------------------------------------------------------------------
ss_d = pd.to_numeric(df["social_studies_exam_score"], errors="coerce")
over = ss_d > 100
print("\n\n(b) SOCIAL STUDIES EXAM SCORE")
print(f"values > 100 : {int(over.sum())} of {int(ss_d.notna().sum())} "
      f"({100*over.mean():.1f}%)")
print(f"non-integer  : {int((ss_d.notna() & (ss_d % 1 != 0)).sum())}")
print(f"unparseable  : {int(ss_d.isna().sum() - df['social_studies_exam_score'].isna().sum())}")
if SCHOOL_COL in df.columns:
    by = (pd.DataFrame({"school": df[SCHOOL_COL], "over100": over, "ss": ss_d})
          .groupby("school")
          .agg(n=("ss", "size"), n_over_100=("over100", "sum"),
               median=("ss", "median"), max=("ss", "max")))
    by["pct_over_100"] = (100 * by["n_over_100"] / by["n"]).round(1)
    print("\nby school:")
    print(by.to_string())
    by.to_csv(OUT / "social_studies_by_school.csv")
print("""
HOW TO READ THIS:
  * values >100 concentrated in ONE school, its median near 2x the others
      -> that school marked out of 200. Set action = "rescale".
  * values >100 scattered thinly across schools
      -> entry errors. Set action = "invalidate".
  * values >100 a large share everywhere
      -> the column is unreliable. Keep action = "exclude" and say so.""")
diag_rows.append({"check": "social_studies_over_100", "n_over_100": int(over.sum()),
                  "pct_over_100": round(100*float(over.mean()), 1)})

# ---------------------------------------------------------------------
# (c) fractional counts: imputed upstream?
# ---------------------------------------------------------------------
print("\n\n(c) FRACTIONAL VALUES IN COUNT COLUMNS")
for col, spec in COUNT_COLS.items():
    if col not in df.columns:
        continue
    raw_ = df[col].astype(str).str.strip().str.lower().replace(
        {k.lower(): str(v) for k, v in spec["strings"].items()})
    num = pd.to_numeric(raw_, errors="coerce")
    frac = num.notna() & (num % 1 != 0)
    toohigh = num > spec["max"]
    unparse = num.isna() & df[col].notna()
    print(f"{col}:")
    print(f"   fractional   : {int(frac.sum()):4d}  e.g. {sorted(num[frac].unique())[:8]}")
    print(f"   above max {spec['max']:<2} : {int(toohigh.sum()):4d}  "
          f"e.g. {sorted(num[toohigh].unique())[:8]}")
    print(f"   unparseable  : {int(unparse.sum()):4d}  "
          f"e.g. {sorted(df[col][unparse].astype(str).unique())[:5]}")
    if frac.sum():
        whole = num.notna() & ~frac
        print(f"   dropout rate, fractional vs whole-number rows: "
              f"{100*y[frac].mean():.1f}% vs {100*y[whole].mean():.1f}%")
    diag_rows.append({"check": f"count_{col}", "n_fractional": int(frac.sum()),
                      "n_above_max": int(toohigh.sum()),
                      "n_unparseable": int(unparse.sum())})
print("""
A count cannot be 0.3. Fractional values look like mean imputation done in
the spreadsheet before the data reached any notebook, which would mean the
raw workbook already carries information from rows that later became test
data. Ask whoever prepared the workbook. The pipeline discards these values
and re-imputes them from each training fold, which is the fold-safe repair
either way. If fractional rows have a very different dropout rate from the
whole-number rows, the upstream imputation was itself outcome-correlated.""")

# ---------------------------------------------------------------------
# (d) did the R01 split put the same pupil on both sides?
# ---------------------------------------------------------------------
print("\n\n(d) DUPLICATE RECORDS ACROSS THE R01 SPLIT")
if n_dup:
    y_all = binarise_target(df_with_dups[TARGET])
    dup_mask = df_with_dups.duplicated(keep=False)
    print(f"{n_dup} surplus rows; {int(dup_mask.sum())} rows belong to a duplicated group")
    print(f"dropouts among duplicated rows: {int(y_all[dup_mask].sum())}")
    try:
        tr_i, te_i = train_test_split(np.arange(len(df_with_dups)),
                                      test_size=TEST_SIZE, random_state=SPLIT_SEED,
                                      stratify=y_all)
        key = df_with_dups.astype(str).agg("|".join, axis=1)
        straddle = len(set(key.iloc[tr_i]) & set(key.iloc[te_i]))
        print(f"re-creating the R01 80/20 split (seed {SPLIT_SEED}, stratified):")
        print(f"   duplicated pupils appearing in BOTH train and test: {straddle}")
        if straddle:
            print("""   Each of these is a pupil the R01 model was tested on after
   training on an identical copy of the same record. That inflates every R01
   test-set figure, and it is a leakage channel separate from the
   fit-before-split problem in GATE-1. Report it in M6.
   (Re-created on the raw row order. If R01 reordered rows before splitting
   the exact count may differ, but the risk is the same.)""")
        diag_rows.append({"check": "duplicates_straddling_R01_split",
                          "n_duplicate_rows": n_dup, "n_straddling": straddle})
    except Exception as e:
        print("could not re-create the split:", e)
else:
    print("no duplicates")

pd.DataFrame(diag_rows).to_csv(OUT / "data_integrity_diagnostics.csv", index=False)
print("\n-> data_integrity_diagnostics.csv")

# ---------------------------------------------------------------------
# (e) are the EXAM SCORES also recorded after the pupil left?
# ---------------------------------------------------------------------
# Term 3 attendance turned out to record the outcome. The same question
# applies to exam scores: if they come from end-of-year exams, pupils who left
# before term 3 could not have sat them — so either they have no real score,
# or a score was filled in afterwards.
print("\n\n(e) EXAM SCORES OF PUPILS WHO HAD ALREADY LEFT")
left_early = zero                       # the 48 with 0% term-3 attendance
exam_cols = [c for c in EXAM_SCORE_COLS + ["average_exam_score"] if c in df.columns]
rows = []
for c in exam_cols:
    v = pd.to_numeric(df[c], errors="coerce")
    rows.append({"exam": c,
                 "left_before_term3_median": v[left_early].median(),
                 "stayed_dropout_median": v[~left_early & (y == 1)].median(),
                 "retained_median": v[y == 0].median(),
                 "left_before_term3_missing": int(v[left_early].isna().sum())})
ex = pd.DataFrame(rows)
print(ex.round(1).to_string(index=False))
ex.to_csv(OUT / "exam_scores_of_early_leavers.csv", index=False)
print("""
HOW TO READ THIS:
  * early leavers have exam scores much LOWER than everyone else, with none
    missing -> scores may have been filled in (e.g. zero or a low default)
    after they left. Then exam scores leak the outcome too.
  * early leavers' scores look like other dropouts' scores -> the exams
    probably came from terms 1-2, before they left. Exam scores are fine.
Ask: WHICH TERM are these exam scores from? If term 3 or end-of-year, the
early leavers could not have sat them.""")
diag_rows.append({"check": "exam_scores_early_leavers",
                  "n_left_before_term3": int(left_early.sum())})
pd.DataFrame(diag_rows).to_csv(OUT / "data_integrity_diagnostics.csv", index=False)

(a) TERM 3 ATTENDANCE OF ZERO vs OUTCOME
dropout_label      dropout  retained  All
term_3_attendance                        
term 3 = 0%             48         0   48
term 3 > 0%             38       895  933
All                     86       895  981

48 of 48 zero-attendance pupils are dropouts (100%)
48 of 86 dropouts have zero term-3 attendance (56%)

dropout rate by term-3 attendance band:
   term 3 in [    0,     0]: n=  48  dropout rate 100.0%
   term 3 in [   50,    80]: n= 286  dropout rate  12.6%
   term 3 in [   80,   101]: n= 670  dropout rate   0.3%

*** STRONG SIGNAL OF TEMPORAL LEAKAGE ***
Nearly every pupil with zero term-3 attendance is a dropout. That is what you
would see if term 3 was recorded after they had already left. If so,
term_3_attendance measures the outcome rather than predicting it, and a large
part of the 0.99 AUC-PR is the model reading the answer.

This is a question for whoever designed the instrument, not a coding
question: WHEN was dropout status rec

In [26]:
# ---- 7. income category check (Q3) --------------------------------------
# The examiner raised family_income_level specifically: engineered_data.csv
# held both "High" and a misspelled "Hgh", which a label encoder treats as two
# levels. Every OTHER categorical is audited comprehensively in 7b below.
col = SOCIOECONOMIC_COLS["family_income"]
if col in df.columns:
    vc = df[col].value_counts(dropna=False)
    print(f"{col} — as recorded:")
    print(vc.to_string())
    canon_map = CATEGORY_CANONICAL.get(col, {})
    seen = {str(v).strip() for v in vc.index if pd.notna(v)}
    unhandled = sorted(v for v in seen
                       if v.lower() not in canon_map and v.lower() not in NA_TOKENS)
    if unhandled:
        print(f"  *** not handled by config: {unhandled} ***")
    targets = sorted({v for v in canon_map.values() if v is not None})
    print(f"  canonical categories : {targets}")
    print(f"  non-response -> NaN  : "
          f"{sorted(v for v in seen if v.lower() in NA_TOKENS)}")
    print(f"  ordinal map          : {ORDINAL_MAPS.get(col)}")
    print("  'Hgh' is merged into 'High'; 'Don't know' becomes missing with its "
          "own indicator rather than being placed at 'Medium' as R01 did.")

family_income_level — as recorded:
family_income_level
Medium        555
High          260
Low           124
Don't know     39
Hgh             3
  canonical categories : ['High', 'Low', 'Medium']
  non-response -> NaN  : ["Don't know"]
  ordinal map          : {'Low': 0, 'Medium': 1, 'High': 2}
  'Hgh' is merged into 'High'; 'Don't know' becomes missing with its own indicator rather than being placed at 'Medium' as R01 did.


In [27]:
# ---- 7b. CATEGORY AUDIT — run this before you trust any ordinal map ----
# Prints every categorical value and flags anything config.ORDINAL_MAPS
# misses. An unmapped category becomes NaN and is then imputed away, so a
# silent gap here loses part of a variable without telling anyone.
problems = audit_categories(df)
if len(problems):
    problems.to_csv(OUT / "category_audit_problems.csv", index=False)
    print("\n-> category_audit_problems.csv written. Fix config.py, re-run "
          "this notebook, and only then continue to Notebook 2.")
else:
    print("\nORDINAL_MAPS and NOMINAL_COLS cover every observed value.")


school_code  [nominal]  4 distinct
    WEWE                                             481 
    KNU_JHS                                          329 
    SHI                                               92 
    AYED_RC                                           79 

geographic_zone  [nominal]  2 distinct
    Rural                                            560 
    Peri-urban                                       421 

school_type  [nominal]  2 distinct
    Public                                           889 
    Private                                           92 

gender  [nominal]  2 distinct
    Male                                             513 
    Female                                           468 

class_level  [ORDINAL]  5 distinct
    JHS2                                             365 -> 4
    JHS1                                             272 -> 3
    P5                                               147 -> 1
    P4                                               12

In [28]:
# ---- 8. cluster audit (GATE-1 iv, Q2, Q18) ------------------------------
if SCHOOL_COL in df.columns:
    g = (df.groupby(SCHOOL_COL)
           .agg(n_pupils=(TARGET, "size"), n_dropout=(TARGET, "sum")))
    g["dropout_rate_pct"] = (100 * g["n_dropout"] / g["n_pupils"]).round(1)
    g["share_of_sample_pct"] = (100 * g["n_pupils"] / len(df)).round(1)
    g = g.sort_values("n_pupils", ascending=False)
    g.to_csv(OUT / "school_cluster_audit.csv")
    print(g.to_string())
    print(f"\neffective cluster count : {g.shape[0]} schools")
    print(f"largest school          : {g['share_of_sample_pct'].iloc[0]:.0f}% of the sample")
    print(f"dropout rate spread     : {g['dropout_rate_pct'].min():.1f}% "
          f"to {g['dropout_rate_pct'].max():.1f}%")
    print(f"\nconfig.SCHOOL_HANDLING = {SCHOOL_HANDLING!r}")
    print("M10/M18 must name this school count, not the metropolitan area.")
else:
    print(f"'{SCHOOL_COL}' not present — verify the column name in config.py")

             n_pupils  n_dropout  dropout_rate_pct  share_of_sample_pct
school_code                                                            
WEWE              481         50              10.4                 49.0
KNU_JHS           329         11               3.3                 33.5
SHI                92         19              20.7                  9.4
AYED_RC            79          6               7.6                  8.1

effective cluster count : 4 schools
largest school          : 49% of the sample
dropout rate spread     : 3.3% to 20.7%

config.SCHOOL_HANDLING = 'drop'
M10/M18 must name this school count, not the metropolitan area.


In [29]:
# ---- 9. save -------------------------------------------------------------
# cleaned_data.csv is intentionally UNIMPUTED and UNENCODED. Every fitted
# transform happens in pipeline.preprocess_inside_fold(). If you find
# yourself writing an "engineered_data.csv" for modelling, stop: that file
# is what failed GATE-1.
df.to_csv(CLEANED_CSV, index=False)
# NO pupil-level snapshot is written into results/. results/ is committed or
# uploaded, and cleaned_data.csv is 1000 identifiable pupil records — ethics
# approval HuSSREC/AP/543/VOL. 5 does not permit sharing it. What goes into
# results/ is the SHAPE of the data (the cascade, missingness, audits), never
# the rows.
pd.DataFrame([{"n_rows": len(df), "n_columns": df.shape[1],
               "n_positive": int(df[TARGET].sum()),
               "base_rate_pct": round(100*float(df[TARGET].mean()), 2),
               "columns": "; ".join(df.columns)}]).to_csv(
    OUT / "cleaned_data_shape.csv", index=False)

write_manifest(OUT, {
    "notebook": "01_cleaning",
    "raw_shape": [int(N_RAW_ROWS), int(N_RAW_COLS)],
    "cleaned_shape": [int(df.shape[0]), int(df.shape[1])],
    "n_positive": int(df[TARGET].sum()),
    "rows_with_missing": int(rows_any),
    "attendance_out_of_range": int(out_df["n_above_100"].sum() + out_df["n_below_0"].sum()) if len(out_df) else 0,
    "columns_restored_vs_substring_rule": restored,
    "dropped": {"leakage": leak, "identifier": ident,
                "empty": empty, "near_unique_text": near_unique},
})

print(f"saved  -> {CLEANED_CSV}")
print(f"shape  : {df.shape} (including target)")
print(f"\nNOT imputed, NOT encoded, NOT scaled — by design. "
      f"Predictors are assembled in-fold.")
print("\nNEXT: Notebook 2 (EDA on the training pool only).")

saved  -> /content/drive/MyDrive/Ghana_Dropout_Project_R02/data-processed/cleaned_data.csv
shape  : (981, 42) (including target)

NOT imputed, NOT encoded, NOT scaled — by design. Predictors are assembled in-fold.

NEXT: Notebook 2 (EDA on the training pool only).


---

## Before you submit

Drive has already saved everything — nothing to push. But two things still
have to happen before submission, and neither is automatic.


In [30]:
# ---- what this run produced, and what is still owed ----
import os, sys
from pathlib import Path

try:
    latest = sorted(Path(OUT).parent.glob("*"))[-1]
    files = sorted(p.relative_to(OUT).as_posix() for p in Path(OUT).rglob("*")
                   if p.is_file())
    print(f"run directory : {Path(OUT).relative_to(REPO)}")
    print(f"files written : {len(files)}")
    for f in files:
        print("   ", f)
except Exception as e:
    print("no run directory recorded in this session:", e)

print("""
────────────────────────────────────────────────────────────────────
STILL OWED BEFORE SUBMISSION — neither happens by itself

1. UPLOAD THE PROJECT TO GITHUB, ONCE.
   Q4 failed because the repository was not runnable from a clone. Drive
   is fine for working; the repo is the deliverable. When the analysis is
   finished, drag the whole project folder into GitHub in one upload —
   EXCEPT data-raw/ and anything holding pupil rows. The notebooks resolve
   paths relative to the project root, so they run from a clone unchanged.

   Do NOT upload:  data-raw/, data-processed/cleaned_data.csv,
                   any *_snapshot.csv
   DO upload:      config.py, losses.py, pipeline.py, notebooks/,
                   requirements.txt, README.md, and all of results/

2. SET config.FREEZE_TAG BEFORE SCORING THE TEST SET.
   Without git there is no commit hash to anchor the freeze to. Put a
   fixed dated string in config.py — e.g. "R02-freeze-2026-09-25-1430" —
   at the moment you freeze the configuration, and never revise it.
   Notebook 8 refuses to score the test set until it is set.
────────────────────────────────────────────────────────────────────""")


run directory : results/notebook01_cleaning/20260921T134433Z_records
files written : 12
    RUN_MANIFEST.json
    attendance_range_audit.csv
    cleaned_data_shape.csv
    column_cascade.csv
    data_integrity_diagnostics.csv
    environment_versions.csv
    exam_scores_of_early_leavers.csv
    figures/missingness.png
    missingness_by_column.csv
    pip_freeze.txt
    school_cluster_audit.csv
    social_studies_by_school.csv

────────────────────────────────────────────────────────────────────
STILL OWED BEFORE SUBMISSION — neither happens by itself

1. UPLOAD THE PROJECT TO GITHUB, ONCE.
   Q4 failed because the repository was not runnable from a clone. Drive
   is fine for working; the repo is the deliverable. When the analysis is
   finished, drag the whole project folder into GitHub in one upload —
   EXCEPT data-raw/ and anything holding pupil rows. The notebooks resolve
   paths relative to the project root, so they run from a clone unchanged.

   Do NOT upload:  data-raw/, dat